In [ ]:
from langchain_ollama import ChatOllama,OllamaEmbeddings

llm = ChatOllama(
    model="minimax-m3:cloud",
    temperature=0
)

In [ ]:
from mem0 import Memory

config = {
    "llm": {
        "provider": "ollama",
        "config": {
            "model": "minimax-m3:cloud",
            "temperature": 0.2,
        },
    },
    "embedder": {
        "provider": "ollama",
        "config": {
            "model": "mxbai-embed-large:latest",
        },
    },
    "vector_store": {
        "provider": "qdrant",
        "config": {
            "collection_name": "ollama_memory_v2",
            "embedding_model_dims": 1024,
        }
    },
    
}


memory = Memory.from_config(config)

In [3]:
def chat_with_memories(message: str, user_id: str = "default_user"):

    relevant_memories = memory.search(
        query=message,
        filters={"user_id": user_id},
        top_k=3,
    )

    memories_str = "\n".join(
        f"- {entry['memory']}"
        for entry in relevant_memories["results"]
    )

    system_prompt = f"""
You are a helpful AI assistant.

User Memories:
{memories_str}
"""

    response = llm.invoke(
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": message},
        ]
    )

    assistant_response = response.content

    memory.add(
        [
            {"role": "user", "content": message},
            {"role": "assistant", "content": assistant_response},
        ],
        user_id=user_id,
    )

    return assistant_response

In [ ]:
import gradio as gr


def chat(user_message, history):
    if not user_message.strip():
        return "Please enter a message."

    if user_message.lower().strip() == "exit":
        return "Goodbye! 👋"

    response = chat_with_memories(user_message)
    return response



def main():
    demo = gr.ChatInterface(
    fn=chat,
    title="🤖 AI Chat with Memory",
    description="Chat with your AI assistant.",
)

    demo.launch()


if __name__ == "__main__":
    main()